In [ ]:
import pandas as pd
import numpy as np
import random

# --- FIX 1: HANDLE THE "m³" SYMBOL ---
# We use encoding='latin1' so Python doesn't crash on the Revit file
try:
    df_revit = pd.read_csv('Revit_Data.csv', encoding='latin1')
    print("SUCCESS: Revit Data Loaded!")
except FileNotFoundError:
    print("ERROR: Please upload 'Revit_Data.csv' first!")

# --- SETUP SIMULATION ---
project_days = 150
dates = pd.date_range(start='2025-11-01', periods=project_days, freq='D')  #Frequency = 'D' → Daily data
iot_dataset = []  #Creates an empty list

# --- SIMULATION LOOP ---
for index, row in df_revit.iterrows():

    # Clean the data safely
    try:
        zone = str(row.get('Comments', 'Unknown Zone'))
        vol_str = str(row['Material: Volume'])
        total_volume = float(vol_str.replace('m³', '').replace('m3', '').strip())
    except:
        continue # Skip bad rows

    avg_daily_usage = total_volume / project_days
    cumulative_usage = 0

    # --- FIX 2: CORRECT INDENTATION (Everything aligned perfectly) ---
    for current_date in dates:

        # 1. Labor & Weather
        labor = np.random.randint(5, 30)
        weather = np.random.choice([0, 1], p=[0.85, 0.15]) # 0=Sun, 1=Rain

        # 2. GIS Logic (Truck Speed)
        if weather == 1:
            speed = np.random.randint(10, 20) # Slow
        else:
            speed = np.random.randint(30, 45) # Fast

        travel_time_min = (15.0 / speed) * 60

        # 3. Civil Logic (Setting Time)
        if travel_time_min > 90:
            quality = "REJECTED"
        elif travel_time_min > 60:
            quality = "WARNING"
        else:
            quality = "GOOD"

        # 4. Usage Logic
        if weather == 1:
            daily_used = 0
        else:
            productivity = 1 + (labor/50)
            daily_used = avg_daily_usage * productivity

        # 5. Sustainability Logic (CO2)
        co2 = daily_used * 288

        # 6. Risk Logic
        cumulative_usage += daily_used
        risk = 1 if cumulative_usage > (total_volume * 0.95) else 0

        # Save Row
        iot_dataset.append([
            current_date, zone, "Concrete", labor, weather,
            int(travel_time_min), quality, round(daily_used, 2),
            round(cumulative_usage, 2), round(co2, 2), risk
        ])

# --- EXPORT DATASET ---
columns = [
    'Date', 'Zone', 'Material', 'Labor_Count', 'Weather_Rain',
    'Truck_Travel_Min', 'Concrete_Quality', 'Daily_Usage_m3',
    'Cumulative_Usage_m3', 'CO2_Emission_kg', 'Shortage_Risk'
]

df_final = pd.DataFrame(iot_dataset, columns=columns)
df_final.to_csv('IoT_Simulation_Output.csv', index=False)

print("\nDONE! The dataset has been generated.")
print("Look for 'IoT_Simulation_Output.csv' in the files tab on the left.")

SUCCESS: Revit Data Loaded!

DONE! The dataset has been generated.
Look for 'IoT_Simulation_Output.csv' in the files tab on the left.
